# 01 — EDA: Spotify Tracks Dataset

**TCC · Sistemas de Informação · UFJF**

Exploração da base [Spotify Tracks Dataset](https://www.kaggle.com/datasets/maharshipandya/-spotify-tracks-dataset).

Este notebook é a **fonte** do visualizador estático em `docs/index.html`.
O HTML já está pré-renderizado — o orientador não precisa executar nada.

## 1. Carregamento e limpeza básica

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')

# Ajuste o caminho conforme o local do CSV
DATA_PATH = 'dataset.csv'  # ou caminho do Kaggle/HF

df = pd.read_csv(DATA_PATH)
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

print('Shape original:', df.shape)
df = df.drop_duplicates(subset=['track_id'])
print('Shape após deduplicar track_id:', df.shape)
df.head()

## 2. Visão geral

In [ ]:
print('Gêneros únicos:', df['track_genre'].nunique())
print('Artistas (combinações) únicos:', df['artists'].nunique())
print('\nValores faltantes:\n', df.isnull().sum()[df.isnull().sum() > 0])
print('\n% explicit:', 100 * df['explicit'].mean())

## 3. Distribuição de gêneros (Top 20)

In [ ]:
genre_counts = df['track_genre'].value_counts()
fig, ax = plt.subplots(figsize=(10, 6))
genre_counts.head(20).sort_values().plot(kind='barh', ax=ax, color='#1DB954')
ax.set_title('Top 20 gêneros por número de tracks')
ax.set_xlabel('Número de tracks')
plt.tight_layout()
plt.show()

## 4. Distribuições das audio features

In [ ]:
audio_features = [
    'danceability', 'energy', 'loudness', 'speechiness',
    'acousticness', 'instrumentalness', 'liveness',
    'valence', 'tempo', 'popularity'
]

fig, axes = plt.subplots(2, 5, figsize=(16, 6))
axes = axes.flatten()
for i, col in enumerate(audio_features):
    sns.histplot(df[col].dropna(), bins=40, ax=axes[i], color='#1DB954')
    axes[i].set_title(col)
    axes[i].set_xlabel('')
fig.suptitle('Distribuição das audio features e popularity', y=1.02)
plt.tight_layout()
plt.show()

## 5. Matriz de correlação

In [ ]:
corr = df[audio_features].corr()
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0, ax=ax, square=True)
ax.set_title('Matriz de correlação (audio features + popularity)')
plt.tight_layout()
plt.show()

## 6. Observações para o TCC

- A base é adequada para recomendação **content-based** (audio features).
- `popularity` é score do Spotify, não feedback de usuário.
- Não há interações usuário–item → avaliação de ranking precisará de estratégia alternativa.
- Features como `instrumentalness` e `speechiness` são fortemente assimétricas.